# 01 — EDA: Telemetria + Apontamentos
**Programa Desenvolver 2026 — Vale S.A.**  
**Desafio:** Análise Avançada de Dados — Antecipação de Alertas Críticos  
**Responsável:** Túlio  
**Referência:** Estudo Guiado CM 2.1 / CM 2.2 / CM 2.3

---

## Objetivo
Desenvolver familiaridade profunda com os dados antes de qualquer modelagem.  
Ao final deste notebook, teremos respondido:
- Qual a estrutura e qualidade da base?
- Qual a frequência e distribuição dos alertas Don't Go?
- Quais equipamentos, turnos e tipos de alarme concentram mais alertas?
- Quais hipóteses levantar para a fase de feature engineering?

## 0. Imports e Configuração

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Adiciona src/ ao path para importar config e limpeza
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from config import (
    TELEMETRIA_FILES, APONTAMENTOS_RAW, ALARMES_REGRAS,
    TELEMETRIA_LIMPA, TARGET, COL_TAG_TELEMETRIA, COL_TIMESTAMP
)
from limpeza import (
    carregar_telemetria_completa, normalizar_criticidade,
    remover_colunas_inuteis, tratar_coluna_valor,
    extrair_features_temporais, calcular_duracao_ciclo_apontamentos,
    relatorio_qualidade, checar_balanceamento
)

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120

print('✅ Imports OK')

---
## 1. Carregamento e Inspeção Inicial (CM 2.1)

### 1.1 Telemetria — Carregamento

In [ ]:
# Carrega todos os 6 meses (Jan–Jun 2025)
# ATENÇÃO: ~37 milhões de linhas — pode demorar 1–2 minutos
print('Carregando telemetria completa...')
df_tel = carregar_telemetria_completa(TELEMETRIA_FILES, verbose=True)

In [ ]:
# Shape e tipos
print(f'Shape: {df_tel.shape}')
print(f'\nTipos de dados:')
print(df_tel.dtypes)
print(f'\nPrimeiras 3 linhas:')
df_tel.head(3)

In [ ]:
# Janela temporal
print(f'Período coberto:')
print(f'  Início : {df_tel[COL_TIMESTAMP].min()}')
print(f'  Fim    : {df_tel[COL_TIMESTAMP].max()}')
print(f'  Dias   : {(df_tel[COL_TIMESTAMP].max() - df_tel[COL_TIMESTAMP].min()).days}')

In [ ]:
# Valores nulos por coluna
nulos = df_tel.isnull().sum()
print('Valores nulos por coluna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else '  ✅ Nenhum valor nulo encontrado.')
print(f'\nDuplicatas: {df_tel.duplicated().sum():,}')

In [ ]:
# Relatório de qualidade completo (CM 2.1 — tabela de estatísticas)
rel_qualidade = relatorio_qualidade(df_tel, nome='Telemetria Bruta')

### 1.2 Distribuição Temporal dos Registros (Figura 2)

In [ ]:
# Volume de eventos por dia
df_tel[COL_TIMESTAMP] = pd.to_datetime(df_tel[COL_TIMESTAMP])
eventos_por_dia = df_tel.groupby(df_tel[COL_TIMESTAMP].dt.date).size()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Volume total por dia
axes[0].plot(eventos_por_dia.index, eventos_por_dia.values, color='steelblue', linewidth=1)
axes[0].set_title('Volume de Eventos de Telemetria por Dia', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Data')
axes[0].set_ylabel('Nº de Eventos')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
axes[0].tick_params(axis='x', rotation=45)

# Volume por hora do dia
vol_hora = df_tel.groupby(df_tel[COL_TIMESTAMP].dt.hour).size()
axes[1].bar(vol_hora.index, vol_hora.values, color='steelblue', edgecolor='white')
axes[1].set_title('Volume de Eventos por Hora do Dia', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hora do Dia')
axes[1].set_ylabel('Nº de Eventos')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.savefig('../relatorio/fig02_distribuicao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figura salva em relatorio/fig02_distribuicao_temporal.png')

---
## 2. Análise da Variável Alvo — Is_Dont_Go (CM 2.2)

### 2.1 Balanceamento de Classes

In [ ]:
checar_balanceamento(df_tel, col_target=TARGET)

### 2.2 Don't Go por Mês (Figura 4)

In [ ]:
# Don't Go por mês — volume e taxa
df_tel['mes_ano'] = df_tel[COL_TIMESTAMP].dt.to_period('M')
dg_mes = df_tel.groupby('mes_ano').agg(
    total=('Is_Dont_Go', 'count'),
    dontgo=('Is_Dont_Go', 'sum')
).reset_index()
dg_mes['taxa_pct'] = dg_mes['dontgo'] / dg_mes['total'] * 100
dg_mes['mes_str'] = dg_mes['mes_ano'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Volume absoluto
axes[0].bar(dg_mes['mes_str'], dg_mes['dontgo'], color='tomato', edgecolor='white')
axes[0].set_title("Don't Go — Volume por Mês", fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel("Nº de Eventos Don't Go")
for i, v in enumerate(dg_mes['dontgo']):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

# Taxa percentual
axes[1].bar(dg_mes['mes_str'], dg_mes['taxa_pct'], color='darkorange', edgecolor='white')
axes[1].set_title("Don't Go — Taxa % por Mês", fontsize=13, fontweight='bold')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel("% Eventos Don't Go")
for i, v in enumerate(dg_mes['taxa_pct']):
    axes[1].text(i, v + 0.001, f'{v:.4f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../relatorio/fig04_dontgo_por_mes.png', dpi=150, bbox_inches='tight')
plt.show()
print(dg_mes[['mes_str', 'total', 'dontgo', 'taxa_pct']].to_string(index=False))

### 2.3 Distribuição por Tipo de Alarme (Figura 3)

In [ ]:
# Top alarmes Don't Go
dg_eventos = df_tel[df_tel[TARGET] == 1]
top_alarmes = dg_eventos['Alarme'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 7))
top_alarmes.sort_values().plot(kind='barh', ax=ax, color='tomato', edgecolor='white')
ax.set_title("Top 15 Alarmes Causadores de Don't Go", fontsize=13, fontweight='bold')
ax.set_xlabel('Nº de Ocorrências')
ax.set_ylabel('Alarme')
for i, v in enumerate(top_alarmes.sort_values().values):
    ax.text(v + 20, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../relatorio/fig03_top_alarmes_dontgo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição por Criticidade
print('Don\'t Go por Criticidade:')
print(dg_eventos['Criticidade'].value_counts())
print('\nDistribuição geral por Criticidade:')
print(df_tel['Criticidade'].value_counts())

### 2.4 Don't Go por Equipamento (TAG)

In [ ]:
# Don't Go por equipamento — valor absoluto e taxa
dg_por_tag = df_tel.groupby(COL_TAG_TELEMETRIA).agg(
    total=('Is_Dont_Go', 'count'),
    dontgo=('Is_Dont_Go', 'sum')
).reset_index()
dg_por_tag['taxa_pct'] = dg_por_tag['dontgo'] / dg_por_tag['total'] * 100
dg_por_tag = dg_por_tag.sort_values('dontgo', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Volume
top = dg_por_tag.head(15)
axes[0].barh(top[COL_TAG_TELEMETRIA], top['dontgo'], color='steelblue', edgecolor='white')
axes[0].set_title("Top 15 Equipamentos — Volume Don't Go", fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nº de Eventos')
axes[0].invert_yaxis()

# Taxa
top_taxa = dg_por_tag.sort_values('taxa_pct', ascending=False).head(15)
axes[1].barh(top_taxa[COL_TAG_TELEMETRIA], top_taxa['taxa_pct'], color='darkorange', edgecolor='white')
axes[1].set_title("Top 15 Equipamentos — Taxa % Don't Go", fontsize=12, fontweight='bold')
axes[1].set_xlabel('Taxa (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../relatorio/fig_dontgo_por_equipamento.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTabela completa Don\'t Go por Equipamento:')
print(dg_por_tag.to_string(index=False))

---
## 3. Limpeza e Normalização (CM 3.1)

In [ ]:
# Aplicar todas as limpezas em sequência com registro ANTES/DEPOIS
print('[1/4] Normalizando Criticidade...')
df_tel = normalizar_criticidade(df_tel)

print('\n[2/4] Removendo colunas inúteis...')
df_tel = remover_colunas_inuteis(df_tel)

print('\n[3/4] Tratando coluna Valor...')
df_tel = tratar_coluna_valor(df_tel)

print('\n[4/4] Extraindo features temporais...')
df_tel = extrair_features_temporais(df_tel)

print(f'\n✅ Shape após limpeza: {df_tel.shape}')

In [ ]:
# Salvar telemetria limpa
df_tel.to_parquet(TELEMETRIA_LIMPA, index=False)
print(f'💾 Telemetria limpa salva em: {TELEMETRIA_LIMPA}')
print(f'   Shape: {df_tel.shape}')

---
## 4. Análise de Features (CM 2.3)

### 4.1 Don't Go por Turno e Hora do Dia (Figura 6)

In [ ]:
# Taxa de Don't Go por hora do dia e dia da semana
taxa_hora = df_tel.groupby('hora_dia')['Is_Dont_Go'].mean() * 100
taxa_diasem = df_tel.groupby('dia_semana')['Is_Dont_Go'].mean() * 100
dias_nome = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(taxa_hora.index, taxa_hora.values, color='steelblue', edgecolor='white')
axes[0].set_title("Taxa de Don't Go por Hora do Dia", fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hora do Dia')
axes[0].set_ylabel("Taxa Don't Go (%)")
axes[0].set_xticks(range(24))

axes[1].bar(range(7), taxa_diasem.values, color='darkorange', edgecolor='white')
axes[1].set_title("Taxa de Don't Go por Dia da Semana", fontsize=13, fontweight='bold')
axes[1].set_xlabel('Dia da Semana')
axes[1].set_ylabel("Taxa Don't Go (%)")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dias_nome)

plt.tight_layout()
plt.savefig('../relatorio/fig06_taxa_hora_diasemana.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Don't Go por Turno

In [ ]:
taxa_turno = df_tel.groupby('turno')['Is_Dont_Go'].agg(['sum', 'count', 'mean'])
taxa_turno.columns = ['Total Don\'t Go', 'Total Eventos', 'Taxa']
taxa_turno['Taxa %'] = (taxa_turno['Taxa'] * 100).round(4)
print('Don\'t Go por Turno:')
print(taxa_turno)

### 4.3 Don't Go por Tipo de Equipamento e Frota

In [ ]:
# Por Tipo (Caminhao vs Escavadeira)
print('Don\'t Go por Tipo:')
print(df_tel.groupby('Tipo')['Is_Dont_Go'].agg(['sum', 'mean']).rename(
    columns={'sum': 'Total', 'mean': 'Taxa'}))

# Por Frota
print('\nDon\'t Go por Frota:')
print(df_tel.groupby('Tag_Frota')['Is_Dont_Go'].agg(['sum', 'mean']).rename(
    columns={'sum': 'Total', 'mean': 'Taxa'}).sort_values('Total', ascending=False))

### 4.4 Comportamento por Operador

In [ ]:
# Operadores com mais Don't Go (hipótese: comportamento do operador influencia alertas)
op_dg = df_tel.groupby('Nome_Operador_Anon')['Is_Dont_Go'].agg(['sum', 'count', 'mean'])
op_dg.columns = ['Total DG', 'Total Eventos', 'Taxa']
op_dg['Taxa %'] = (op_dg['Taxa'] * 100).round(4)
op_dg = op_dg[op_dg['Total Eventos'] > 100]  # filtrar operadores com poucos eventos
op_dg_sorted = op_dg.sort_values('Total DG', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 6))
op_dg_sorted['Total DG'].sort_values().plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title("Top 20 Operadores — Volume Don't Go", fontsize=13, fontweight='bold')
ax.set_xlabel('Nº de Eventos Don\'t Go')
plt.tight_layout()
plt.savefig('../relatorio/fig_dontgo_por_operador.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTotal de operadores analisados (>100 eventos): {len(op_dg)}')

### 4.5 Heatmap de Correlação (Figura 5)

In [ ]:
# Correlação entre features numéricas e o target
cols_num = df_tel.select_dtypes(include=[np.number]).columns.tolist()
print(f'Colunas numéricas: {cols_num}')

if len(cols_num) > 1:
    corr = df_tel[cols_num].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='coolwarm', center=0, ax=ax,
        linewidths=0.5, square=True
    )
    ax.set_title('Heatmap de Correlação — Features Numéricas', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../relatorio/fig05_heatmap_correlacao.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Correlação específica com o target
    print('\nCorrelação com Is_Dont_Go:')
    print(corr[TARGET].sort_values(ascending=False))
else:
    print('Poucas colunas numéricas na base bruta — mais features virão na engenharia.')

---
## 5. EDA — Apontamentos

In [ ]:
# Carrega apontamentos
df_ap = pd.read_parquet(APONTAMENTOS_RAW)
df_ap = calcular_duracao_ciclo_apontamentos(df_ap)

print(f'Shape: {df_ap.shape}')
print(f'\nColunas: {df_ap.columns.tolist()}')
print(f'\nDistribuição de Classe:')
print(df_ap['Classe'].value_counts())
print(f'\nDistribuição de Tipo:')
print(df_ap['Tipo'].value_counts())
print(f'\nDuração dos ciclos (min):')
print(df_ap['duracao_min'].describe().round(2))

In [ ]:
# Duração por Classe
fig, ax = plt.subplots(figsize=(12, 5))
df_ap.boxplot(column='duracao_min', by='Classe', ax=ax, showfliers=False)
ax.set_title('Duração dos Ciclos por Classe de Atividade', fontsize=13, fontweight='bold')
ax.set_xlabel('Classe')
ax.set_ylabel('Duração (minutos)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../relatorio/fig_duracao_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Regras de Negócio — Alarmes (CM 1.1)

In [ ]:
# Carrega catálogo de regras
df_alarm = pd.read_excel(ALARMES_REGRAS)
df_alarm['NIVEL'] = df_alarm['NIVEL'].str.strip().str.title()

print(f'Shape: {df_alarm.shape}')
print(f'\nColunas: {df_alarm.columns.tolist()}')
print(f'\nNíveis de alerta:')
print(df_alarm['NIVEL'].value_counts())
print(f'\nEventos de NÍVEL MUITO ALTO (criticidade máxima):')
muito_alto = df_alarm[df_alarm['NIVEL'].str.lower().str.strip() == 'muito alto']
print(muito_alto['EVENTO'].value_counts().head(20))

In [ ]:
# Quais regras de Nível Muito Alto aparecem na base de telemetria?
alarmes_muito_alto = set(muito_alto['EVENTO'].str.strip().str.lower())
alarmes_na_base = set(df_tel['Alarme'].str.strip().str.lower())
intersecao = alarmes_muito_alto & alarmes_na_base

print(f'Regras Muito Alto no catálogo     : {len(alarmes_muito_alto)}')
print(f'Alarmes únicos na base            : {len(alarmes_na_base)}')
print(f'Intersecção (catálogo ∩ base)     : {len(intersecao)}')
print(f'\nAlarmes Muito Alto presentes na base:')
for a in sorted(intersecao):
    print(f'  - {a}')

---
## 7. Síntese — Hipóteses para Feature Engineering

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         HIPÓTESES LEVANTADAS NA EDA — PRÓXIMOS PASSOS           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  H1: Equipamentos com histórico recente de DG têm maior         ║
║      probabilidade de novo DG nas próximas horas.               ║
║      → Feature: dias_desde_ultimo_dontgo_por_TAG                ║
║                                                                  ║
║  H2: Certos alarmes precedem DG de forma sistemática            ║
║      (ex: Engine Coolant Level antes de DG).                    ║
║      → Feature: contagem_alarme_X_ultimas_N_horas               ║
║                                                                  ║
║  H3: Turno noturno pode concentrar mais DG por menor            ║
║      atenção ou condições de temperatura.                        ║
║      → Feature: turno (dia/noite), hora_dia                     ║
║                                                                  ║
║  H4: Comportamento do operador tem correlação com DG.           ║
║      → Feature: taxa_dontgo_historica_operador                  ║
║                                                                  ║
║  H5: Alarmes do tipo Crítico concentrados em janela             ║
║      curta são precursores de DG.                               ║
║      → Feature: contagem_criticos_ultima_1h_por_TAG             ║
║                                                                  ║
║  ⚠️  DESBALANCEAMENTO: 0.054% positivos (1:1862)               ║
║      → Usar class_weight='balanced' no XGBoost                  ║
║      → Métrica principal: Recall, F1-score, AUC-PR              ║
║      → NÃO usar Accuracy                                        ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Sumário final para o relatório
print('=== SUMÁRIO FINAL DA EDA ===')
print(f'  Total de eventos de telemetria  : {len(df_tel):>15,}')
print(f'  Total de Don\'t Go              : {df_tel[TARGET].sum():>15,}')
print(f'  Taxa geral de Don\'t Go         : {df_tel[TARGET].mean()*100:>14.4f}%')
print(f'  Equipamentos únicos             : {df_tel[COL_TAG_TELEMETRIA].nunique():>15,}')
print(f'  Alarmes únicos                  : {df_tel["Alarme"].nunique():>15,}')
print(f'  Operadores únicos               : {df_tel["Nome_Operador_Anon"].nunique():>15,}')
print(f'  Período coberto                 : Jan–Jun 2025 (6 meses)')
print(f'  Colunas após limpeza            : {df_tel.shape[1]:>15}')
print(f'\n  Próxima etapa: 02_feature_engineering.ipynb (Túlio)')